In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
url = "http://ufcstats.com/statistics/events/completed?page=all"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.content, 'html.parser')

# gets the link for each ufc event
event_links = []
for link in soup.select('a.b-link.b-link_style_black'):
    event_url = link.get('href')
    if event_url and 'event-details' in event_url:
        event_links.append(event_url)


In [2]:
fight_links = []

for event_url in event_links: #gets the links of all the matches from each ufc event
    response = requests.get(event_url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract event date
    date_element = soup.find('li', class_='b-list__box-list-item')
    event_date = date_element.text.split(':')[-1].strip() if date_element else 'N/A'
    
    # Find all fight rows
    for row in soup.find_all('tr', {'class': 'b-fight-details__table-row'}):
        onclick = row.get('onclick', '')
        if 'doNav' in onclick:
            fight_url = onclick.split("'")[1]
            fight_links.append({
                'url': fight_url,
                'event_date': event_date  # Store date with fight URL
            })

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

def scrape_ufc_fight_details(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # --- 1. Extract Meta Data ---
    
    # Helper to extract text based on a label (e.g., "Round:", "Time:")
    def get_meta_field(label_text):
        # Find the label element (e.g., <i class="label">Round:</i>)
        label_el = soup.find('i', class_='b-fight-details__label', string=lambda t: t and label_text in t.strip())
        if label_el:
            # The value is the text immediately following the label tag
            return label_el.next_sibling.strip()
        return ""

    # Existing Method/Referee extraction
    method_tag = soup.find('i', class_='b-fight-details__text-item_first')
    method = method_tag.find('i', style="font-style: normal").text.strip() if method_tag else ''

    referee = ""
    referee_span = soup.select_one('i.b-fight-details__label:-soup-contains("Referee:") + span')
    if referee_span:
        referee = referee_span.text.strip()
    
    # New Meta Fields
    last_round = get_meta_field("Round:")
    time_val = get_meta_field("Time:")
    time_format = get_meta_field("Time format:")

    # Winner Logic
    fighter_status_tag = soup.find('i', class_='b-fight-details__person-status')
    first_fighter_won = None
    if fighter_status_tag:
        status = fighter_status_tag.text.strip()
        first_fighter_won = 1 if status == "W" else 0

    # --- 2. Helper Functions for Parsing ---
    def parse_totals_row(row_data):
        if not row_data or len(row_data) < 20: return None
        f1_name, f2_name = row_data[0], row_data[1]
        return {
            f1_name: {
                "kd": row_data[2], "sig_str": row_data[4], "sig_str_prcnt": row_data[6],
                "total_str": row_data[8], "td": row_data[10], "td_prcnt": row_data[12],
                "sub_att": row_data[14], "rev": row_data[16], "ctrl": row_data[18]
            },
            f2_name: {
                "kd": row_data[3], "sig_str": row_data[5], "sig_str_prcnt": row_data[7],
                "total_str": row_data[9], "td": row_data[11], "td_prcnt": row_data[13],
                "sub_att": row_data[15], "rev": row_data[17], "ctrl": row_data[19]
            }
        }

    def parse_sig_row(row_data):
        if not row_data or len(row_data) < 18: return None
        f1_name, f2_name = row_data[0], row_data[1]
        return {
            f1_name: {
                "sig_str": row_data[2], "sig_str_prcnt": row_data[4], "head": row_data[6],
                "body": row_data[8], "leg": row_data[10], "distance": row_data[12],
                "clinch": row_data[14], "ground": row_data[16]
            },
            f2_name: {
                "sig_str": row_data[3], "sig_str_prcnt": row_data[5], "head": row_data[7],
                "body": row_data[9], "leg": row_data[11], "distance": row_data[13],
                "clinch": row_data[15], "ground": row_data[17]
            }
        }

    # --- 3. Extract Raw Rows ---
    tables = soup.find_all('table')

    def extract_raw_rows(table):
        extracted_data = []
        rows = table.find_all('tr')
        for row in rows:
            cols = row.find_all('p', class_='b-fight-details__table-text')
            if not cols: continue
            row_data = [col.get_text(strip=True) for col in cols]
            extracted_data.append(row_data)
        return extracted_data

    # --- 4. Construct JSON Structure ---
    data = {
        "meta": {
            "method": method,
            "referee": referee,
            "round": last_round,
            "time": time_val,
            "time_format": time_format,
            "first_fighter_won": first_fighter_won
        },
        "totals": {},
        "significant_strikes": {}
    }

    if len(tables) >= 4:
        totals_match = extract_raw_rows(tables[0])
        totals_rounds = extract_raw_rows(tables[1])
        if totals_match:
            data["totals"]["match"] = parse_totals_row(totals_match[0])
        data["totals"]["rounds"] = []
        for i, row in enumerate(totals_rounds):
            parsed = parse_totals_row(row)
            if parsed:
                data["totals"]["rounds"].append({"round": i + 1, **parsed})

        sig_match = extract_raw_rows(tables[2])
        sig_rounds = extract_raw_rows(tables[3])
        if sig_match:
            data["significant_strikes"]["match"] = parse_sig_row(sig_match[0])
        data["significant_strikes"]["rounds"] = []
        for i, row in enumerate(sig_rounds):
            parsed = parse_sig_row(row)
            if parsed:
                data["significant_strikes"]["rounds"].append({"round": i + 1, **parsed})

    return json.dumps(data, indent=4)

# Test
url = 'http://ufcstats.com/fight-details/81b84b34f0357fa5'
print(scrape_ufc_fight_details(url))

Final Version

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

# --- 1. Define the Scrape Function ---
def scrape_ufc_fight_details(url):
    try:
        # Standard request with timeout
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # --- Extract Meta Data ---
        def get_meta_field(label_text):
            label_el = soup.find('i', class_='b-fight-details__label', string=lambda t: t and label_text in t.strip())
            if label_el:
                return label_el.next_sibling.strip()
            return ""

        method_tag = soup.find('i', class_='b-fight-details__text-item_first')
        method = method_tag.find('i', style="font-style: normal").text.strip() if method_tag else ''

        referee = ""
        referee_span = soup.select_one('i.b-fight-details__label:-soup-contains("Referee:") + span')
        if referee_span:
            referee = referee_span.text.strip()
        
        last_round = get_meta_field("Round:")
        time_val = get_meta_field("Time:")
        time_format = get_meta_field("Time format:")

        fighter_status_tag = soup.find('i', class_='b-fight-details__person-status')
        first_fighter_won = None
        if fighter_status_tag:
            status = fighter_status_tag.text.strip()
            first_fighter_won = 1 if status == "W" else 0

        # --- Helper Functions for Parsing ---
        def parse_totals_row(row_data):
            if not row_data or len(row_data) < 20: return None
            f1_name, f2_name = row_data[0], row_data[1]
            return {
                f1_name: {
                    "kd": row_data[2], "sig_str": row_data[4], "sig_str_prcnt": row_data[6],
                    "total_str": row_data[8], "td": row_data[10], "td_prcnt": row_data[12],
                    "sub_att": row_data[14], "rev": row_data[16], "ctrl": row_data[18]
                },
                f2_name: {
                    "kd": row_data[3], "sig_str": row_data[5], "sig_str_prcnt": row_data[7],
                    "total_str": row_data[9], "td": row_data[11], "td_prcnt": row_data[13],
                    "sub_att": row_data[15], "rev": row_data[17], "ctrl": row_data[19]
                }
            }

        def parse_sig_row(row_data):
            if not row_data or len(row_data) < 18: return None
            f1_name, f2_name = row_data[0], row_data[1]
            return {
                f1_name: {
                    "sig_str": row_data[2], "sig_str_prcnt": row_data[4], "head": row_data[6],
                    "body": row_data[8], "leg": row_data[10], "distance": row_data[12],
                    "clinch": row_data[14], "ground": row_data[16]
                },
                f2_name: {
                    "sig_str": row_data[3], "sig_str_prcnt": row_data[5], "head": row_data[7],
                    "body": row_data[9], "leg": row_data[11], "distance": row_data[13],
                    "clinch": row_data[15], "ground": row_data[17]
                }
            }

        # --- Extract Raw Rows ---
        tables = soup.find_all('table')

        def extract_raw_rows(table):
            extracted_data = []
            rows = table.find_all('tr')
            for row in rows:
                cols = row.find_all('p', class_='b-fight-details__table-text')
                if not cols: continue
                row_data = [col.get_text(strip=True) for col in cols]
                extracted_data.append(row_data)
            return extracted_data

        # --- Construct JSON Structure ---
        data = {
            "meta": {
                "method": method,
                "referee": referee,
                "round": last_round,
                "time": time_val,
                "time_format": time_format,
                "first_fighter_won": first_fighter_won
            },
            "totals": {},
            "significant_strikes": {}
        }

        if len(tables) >= 4:
            totals_match = extract_raw_rows(tables[0])
            totals_rounds = extract_raw_rows(tables[1])
            if totals_match:
                data["totals"]["match"] = parse_totals_row(totals_match[0])
            data["totals"]["rounds"] = []
            for i, row in enumerate(totals_rounds):
                parsed = parse_totals_row(row)
                if parsed:
                    data["totals"]["rounds"].append({"round": i + 1, **parsed})

            sig_match = extract_raw_rows(tables[2])
            sig_rounds = extract_raw_rows(tables[3])
            if sig_match:
                data["significant_strikes"]["match"] = parse_sig_row(sig_match[0])
            data["significant_strikes"]["rounds"] = []
            for i, row in enumerate(sig_rounds):
                parsed = parse_sig_row(row)
                if parsed:
                    data["significant_strikes"]["rounds"].append({"round": i + 1, **parsed})

        return json.dumps(data)
    
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

# --- 2. Main Execution Block ---

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

print("Step 1: Fetching Event Links...")
url = "http://ufcstats.com/statistics/events/completed?page=all"
response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.content, 'html.parser')

event_links = []
for link in soup.select('a.b-link.b-link_style_black'):
    event_url = link.get('href')
    if event_url and 'event-details' in event_url:
        event_links.append(event_url)

print(f"Found {len(event_links)} events. Gathering fight links...")

fight_links = []
count = 0
for event_url in event_links:
    try:
        response = requests.get(event_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        date_element = soup.find('li', class_='b-list__box-list-item')
        event_date = date_element.text.split(':')[-1].strip() if date_element else 'N/A'
        
        for row in soup.find_all('tr', {'class': 'b-fight-details__table-row'}):
            onclick = row.get('onclick', '')
            if 'doNav' in onclick:
                fight_url = onclick.split("'")[1]
                fight_links.append({
                    'url': fight_url,
                    'event_date': event_date
                })
        
        count += 1
        if count % 50 == 0:
            print(f"Processed {count}/{len(event_links)} events...")
            
    except Exception as e:
        print(f"Error reading event {event_url}: {e}")
        continue

print(f"Total fights found: {len(fight_links)}")
print("Step 2: Starting detailed scrape...")

# --- 3. Scrape Every Fight and Build Dataset ---
final_data = []
count = 0
total_fights = len(fight_links)

for fight in fight_links:
    details_json = scrape_ufc_fight_details(fight['url'])
    
    if details_json:
        details = json.loads(details_json)
        
        fighter_1 = "Unknown"
        fighter_2 = "Unknown"
        winner = "Unknown"
        
        if "totals" in details and "match" in details["totals"] and details["totals"]["match"]:
            keys = list(details["totals"]["match"].keys())
            if len(keys) >= 2:
                fighter_1 = keys[0]
                fighter_2 = keys[1]
                
                fw_won = details['meta']['first_fighter_won']
                if fw_won == 1:
                    winner = fighter_1
                elif fw_won == 0:
                    winner = fighter_2
                else:
                    winner = "Draw/NC"

        row = {
            "event_date": fight['event_date'],
            "fighter_1": fighter_1,
            "fighter_2": fighter_2,
            "winner": winner,
            "method": details['meta'].get('method'),
            "round": details['meta'].get('round'),
            "time": details['meta'].get('time'),
            "fight_url": fight['url'],
            "details_json": details_json
        }
        final_data.append(row)
    
    count += 1
    if count % 100 == 0:
        print(f"Scraped {count}/{total_fights} fights...")

# --- 4. Export to CSV ---
print("Scraping complete. Saving to CSV...")
df = pd.DataFrame(final_data)

output_path = '/kaggle/working/ufc_fight_data.csv'
df.to_csv(output_path, index=False)

print(f"File saved successfully at: {output_path}")